In [1]:
import pandas as pd
import numpy as np
from faker import Faker
import random
from datetime import datetime, timedelta

fake = Faker('en_IN')
np.random.seed(42)
random.seed(42)

N = 650

# --- Reference Data ---
industries = ['SaaS', 'FinTech', 'Healthcare', 'E-Commerce', 'Manufacturing',
              'Logistics', 'EdTech', 'Retail', 'Real Estate', 'IT Services']

lead_sources = ['Inbound', 'Referral', 'Cold Outreach', 'LinkedIn', 'Event',
                'Partner', 'Website', 'Cold Email', 'inbound', 'REFERRAL',
                'linkedin', 'cold outreach']

# Intentionally messy stage names
stage_pool = ['Won', 'WON', 'won', 'closed won', 'Closed Won', 'wonnn',
              'Lost', 'lost', 'LOST', 'Closed Lost',
              'Proposal Sent', 'proposal sent', 'PROPOSAL SENT',
              'Qualified', 'qualified', 'QUALIFIED',
              'Discovery', 'discovery',
              'Negotiation', 'negotiation', 'NEGOTIATION',
              'Prospecting', 'prospecting']

reps = ['Riya Sharma', 'Aman Verma', 'Priya Nair', 'Karan Mehta',
        'Sonal Joshi', 'Deepak Rao', 'Neha Gupta', 'Arjun Pillai']

regions_pool = ['North', 'South', 'East', 'West', 'NORTH', 'south',
                'SOUTH', 'north', 'East India', 'west', 'WEST']

next_steps_pool = ['Schedule Demo', 'Send Proposal', 'Follow Up Call',
                   'Contract Review', 'Stakeholder Meeting', 'Technical Eval',
                   'Pricing Discussion', None]

def random_date(start_year=2022, end_year=2024):
    start = datetime(start_year, 1, 1)
    end = datetime(end_year, 12, 31)
    return start + timedelta(days=random.randint(0, (end - start).days))

def format_date_messy(dt):
    if dt is None:
        return None
    formats = ['%Y-%m-%d', '%d/%m/%Y', '%m-%d-%Y', '%d %b %Y', '%B %d, %Y']
    return dt.strftime(random.choice(formats))

# --- Generate Base Data ---
deal_ids = [f'DEAL-{str(i).zfill(4)}' for i in range(1, N + 1)]
company_names = [fake.company() for _ in range(N)]
created_dates = [random_date(2022, 2023) for _ in range(N)]
close_dates = []

for cd in created_dates:
    if random.random() < 0.12:  # ~12% null close dates
        close_dates.append(None)
    else:
        close_dates.append(cd + timedelta(days=random.randint(10, 180)))

deal_values = []
for _ in range(N):
    r = random.random()
    if r < 0.10:       # null
        deal_values.append(None)
    elif r < 0.14:     # negative (dirty data)
        deal_values.append(round(random.uniform(-50000, -500), 2))
    elif r < 0.17:     # outlier
        deal_values.append(round(random.uniform(500000, 2000000), 2))
    else:
        deal_values.append(round(random.uniform(5000, 250000), 2))

probabilities = [random.choice([10, 20, 30, 40, 50, 60, 70, 80, 90, 100]) for _ in range(N)]

next_steps = [random.choice(next_steps_pool) for _ in range(N)]
# Force ~13% nulls in next_step
for i in random.sample(range(N), int(N * 0.13)):
    next_steps[i] = None

df = pd.DataFrame({
    'deal_id':      deal_ids,
    'company_name': company_names,
    'industry':     [random.choice(industries) for _ in range(N)],
    'lead_source':  [random.choice(lead_sources) for _ in range(N)],
    'deal_value':   deal_values,
    'stage':        [random.choice(stage_pool) for _ in range(N)],
    'created_date': [format_date_messy(d) for d in created_dates],
    'close_date':   [format_date_messy(d) for d in close_dates],
    'sales_rep':    [random.choice(reps) for _ in range(N)],
    'region':       [random.choice(regions_pool) for _ in range(N)],
    'next_step':    next_steps,
    'probability':  probabilities,
})

# --- Inject Duplicates (~7%) ---
dup_indices = random.sample(range(N), int(N * 0.07))
dup_rows = df.iloc[dup_indices].copy()
df = pd.concat([df, dup_rows], ignore_index=True)
df = df.sample(frac=1, random_state=42).reset_index(drop=True)  # shuffle

# --- Save ---
df.to_csv('sales_crm_raw.csv', index=False)

print(f"Dataset shape: {df.shape}")
print("\nFirst 10 rows:")
print(df.head(10).to_string())

Dataset shape: (695, 12)

First 10 rows:
     deal_id            company_name     industry lead_source  deal_value          stage        created_date   close_date     sales_rep region            next_step  probability
0  DEAL-0382               Dugar PLC   E-Commerce    Referral   196753.32    Negotiation         11 Jun 2022   24/09/2022    Priya Nair  north   Pricing Discussion           70
1  DEAL-0236          Mahajan-Bhatia      FinTech    REFERRAL    21486.27     Closed Won         11 Mar 2022   07-23-2022    Aman Verma   west                 None           50
2  DEAL-0258          Chawla-Thakkar  IT Services    REFERRAL   215097.60  Proposal Sent          03-18-2023   2023-08-14   Karan Mehta  SOUTH  Stakeholder Meeting           30
3  DEAL-0339              Amble-Bedi         SaaS     Partner   138042.52          wonnn         07 Aug 2023  15 Oct 2023    Priya Nair   WEST   Pricing Discussion           80
4  DEAL-0320             Johal-Sodhi  Real Estate     Inbound   156846.31 

# Remove Duplicates

In [3]:
import pandas as pd

df = pd.read_csv('sales_crm_raw.csv')

print("Shape before dedup:", df.shape)
print("Duplicate deal_id count:", df.duplicated(subset='deal_id').sum())

# Keep first occurrence of each deal_id
df = df.drop_duplicates(subset='deal_id', keep='first')

print("Shape after dedup:", df.shape)

Shape before dedup: (695, 12)
Duplicate deal_id count: 45
Shape after dedup: (650, 12)


# Fix Stage Names

In [4]:
print("Current unique stage values:")
print(df['stage'].value_counts())

Current unique stage values:
stage
Proposal Sent    36
Closed Lost      36
Negotiation      35
QUALIFIED        34
negotiation      34
Prospecting      34
prospecting      33
Closed Won       32
qualified        31
Discovery        28
discovery        28
closed won       27
won              27
PROPOSAL SENT    26
NEGOTIATION      25
WON              25
LOST             25
Lost             25
wonnn            24
Qualified        24
Won              24
proposal sent    21
lost             16
Name: count, dtype: int64


In [5]:
# Step 1 - Normalize: strip whitespace, lowercase
df['stage'] = df['stage'].str.strip().str.lower()

# Step 2 - Map all dirty variants to clean standard names
stage_mapping = {
    'won': 'Closed Won',
    'closed won': 'Closed Won',
    'wonnn': 'Closed Won',
    'lost': 'Closed Lost',
    'closed lost': 'Closed Lost',
    'proposal sent': 'Proposal Sent',
    'qualified': 'Qualified',
    'discovery': 'Discovery',
    'negotiation': 'Negotiation',
    'prospecting': 'Prospecting'
}

df['stage'] = df['stage'].map(stage_mapping)

print("\nCleaned unique stage values:")
print(df['stage'].value_counts())
print("\nAny unmapped stages (NaN)?", df['stage'].isna().sum())


Cleaned unique stage values:
stage
Closed Won       159
Closed Lost      102
Negotiation       94
Qualified         89
Proposal Sent     83
Prospecting       67
Discovery         56
Name: count, dtype: int64

Any unmapped stages (NaN)? 0


# Standardize Region & Lead Source

In [6]:
# Check current mess
print("Region unique values:")
print(df['region'].value_counts())

print("\nLead source unique values:")
print(df['lead_source'].value_counts())

Region unique values:
region
North         69
west          68
West          67
WEST          65
north         59
south         57
South         57
East India    55
SOUTH         52
East          51
NORTH         50
Name: count, dtype: int64

Lead source unique values:
lead_source
REFERRAL         64
cold outreach    62
Cold Outreach    62
inbound          61
Website          57
LinkedIn         53
Inbound          53
linkedin         51
Partner          50
Cold Email       48
Event            46
Referral         43
Name: count, dtype: int64


In [7]:
# Standardize region
df['region'] = df['region'].str.strip().str.title()
region_mapping = {
    'East India': 'East',
    'North': 'North',
    'South': 'South',
    'East': 'East',
    'West': 'West'
}
df['region'] = df['region'].map(region_mapping)

# Standardize lead_source
df['lead_source'] = df['lead_source'].str.strip().str.title()

print("\nCleaned Region values:")
print(df['region'].value_counts())

print("\nCleaned Lead Source values:")
print(df['lead_source'].value_counts())


Cleaned Region values:
region
West     200
North    178
South    166
East     106
Name: count, dtype: int64

Cleaned Lead Source values:
lead_source
Cold Outreach    124
Inbound          114
Referral         107
Linkedin         104
Website           57
Partner           50
Cold Email        48
Event             46
Name: count, dtype: int64


# Fix Deal Values

In [8]:
print("Null deal_values:", df['deal_value'].isna().sum())
print("Negative deal_values:", (df['deal_value'] < 0).sum())
print("\nDeal value stats before fix:")
print(df['deal_value'].describe())

Null deal_values: 59
Negative deal_values: 19

Deal value stats before fix:
count    5.910000e+02
mean     1.545922e+05
std      2.136367e+05
min     -4.966502e+04
25%      6.132368e+04
50%      1.319350e+05
75%      1.872929e+05
max      1.990225e+06
Name: deal_value, dtype: float64


In [9]:
# Fix negatives first — take absolute value
df['deal_value'] = df['deal_value'].abs()

# Fill nulls with mean grouped by industry + lead_source
df['deal_value'] = df.groupby(['industry', 'lead_source'])['deal_value'] \
                     .transform(lambda x: x.fillna(x.mean()))

# If any nulls remain (group had all nulls), fill with global median
remaining_nulls = df['deal_value'].isna().sum()
print(f"\nRemaining nulls after group fill: {remaining_nulls}")
df['deal_value'] = df['deal_value'].fillna(df['deal_value'].median())

print("\nDeal value stats after fix:")
print(df['deal_value'].describe())


Remaining nulls after group fill: 0

Deal value stats after fix:
count    6.500000e+02
mean     1.561953e+05
std      2.041344e+05
min      8.540800e+02
25%      7.018112e+04
50%      1.304558e+05
75%      1.876359e+05
max      1.990225e+06
Name: deal_value, dtype: float64


# Fix Date Columns

In [10]:
# Check nulls before
print("Null created_date:", df['created_date'].isna().sum())
print("Null close_date:", df['close_date'].isna().sum())

Null created_date: 0
Null close_date: 73


In [11]:
# Parse mixed date formats
df['created_date'] = pd.to_datetime(df['created_date'], dayfirst=False, infer_datetime_format=True, errors='coerce')
df['close_date'] = pd.to_datetime(df['close_date'], dayfirst=False, infer_datetime_format=True, errors='coerce')

print("\nAfter parsing:")
print("Null created_date:", df['created_date'].isna().sum())
print("Null close_date:", df['close_date'].isna().sum())

# Calculate sales cycle length in days (only where both dates exist)
df['sales_cycle_days'] = (df['close_date'] - df['created_date']).dt.days

# Drop rows where sales_cycle_days is negative (close before creation = corrupt record)
bad_rows = (df['sales_cycle_days'] < 0).sum()
print(f"\nRows with negative sales cycle: {bad_rows}")
df = df[~(df['sales_cycle_days'] < 0)]

# Save cleaned file
df.to_csv('sales_crm_clean.csv', index=False)
print(f"\nFinal clean dataset shape: {df.shape}")
print("\nSample of cleaned data:")
print(df[['deal_id','stage','deal_value','region','created_date','close_date','sales_cycle_days']].head(10).to_string())


After parsing:
Null created_date: 514
Null close_date: 544

Rows with negative sales cycle: 0

Final clean dataset shape: (650, 13)

Sample of cleaned data:
     deal_id          stage     deal_value region created_date close_date  sales_cycle_days
0  DEAL-0382    Negotiation  196753.320000  North   2022-06-11 2022-09-24             105.0
1  DEAL-0236     Closed Won   21486.270000   West   2022-03-11        NaT               NaN
2  DEAL-0258  Proposal Sent  215097.600000  South          NaT        NaT               NaN
3  DEAL-0339     Closed Won  138042.520000   West   2023-08-07        NaT               NaN
4  DEAL-0320    Closed Lost  156846.310000  North          NaT        NaT               NaN
5  DEAL-0212     Closed Won  176594.300000  North          NaT        NaT               NaN
6  DEAL-0368     Closed Won   90178.574167  South          NaT        NaT               NaN
7  DEAL-0177      Discovery  121976.250000  South          NaT        NaT               NaN
8  DEAL-0336  

C:\Users\mahaj\AppData\Local\Temp\ipykernel_21164\2512796453.py:2: UserWarning: The argument 'infer_datetime_format' is deprecated and will be removed in a future version. A strict version of it is now the default, see https://pandas.pydata.org/pdeps/0004-consistent-to-datetime-parsing.html. You can safely remove this argument.
  df['created_date'] = pd.to_datetime(df['created_date'], dayfirst=False, infer_datetime_format=True, errors='coerce')
C:\Users\mahaj\AppData\Local\Temp\ipykernel_21164\2512796453.py:3: UserWarning: The argument 'infer_datetime_format' is deprecated and will be removed in a future version. A strict version of it is now the default, see https://pandas.pydata.org/pdeps/0004-consistent-to-datetime-parsing.html. You can safely remove this argument.
  df['close_date'] = pd.to_datetime(df['close_date'], dayfirst=False, infer_datetime_format=True, errors='coerce')
C:\Users\mahaj\AppData\Local\Temp\ipykernel_21164\2512796453.py:3: UserWarning: Parsing dates in %d/%m/%Y 

In [12]:
# Reload raw to see what formats actually exist
df_raw = pd.read_csv('sales_crm_raw.csv')
df_raw = df_raw.drop_duplicates(subset='deal_id', keep='first')

print("Sample of raw created_date values:")
print(df_raw['created_date'].dropna().head(20).tolist())

Sample of raw created_date values:
['11 Jun 2022', '11 Mar 2022', '03-18-2023', '07 Aug 2023', 'November 18, 2022', '13/02/2023', 'September 29, 2023', '15/06/2022', '08-29-2022', 'November 06, 2023', '11 Aug 2022', 'July 02, 2022', '2022-08-22', '2022-08-03', '2022-05-16', '16 May 2023', 'February 12, 2022', '21 Feb 2022', '2022-09-01', '04/06/2023']


In [13]:
from dateutil import parser as dateutil_parser

def parse_date_robust(val):
    if pd.isna(val) or val is None:
        return pd.NaT
    try:
        return dateutil_parser.parse(str(val), dayfirst=False)
    except:
        return pd.NaT

# Apply to cleaned df (reload to avoid stacking errors)
df = pd.read_csv('sales_crm_clean.csv')

df['created_date'] = df['created_date'].apply(parse_date_robust)
df['close_date'] = df['close_date'].apply(parse_date_robust)

print("Null created_date after robust parse:", df['created_date'].isna().sum())
print("Null close_date after robust parse:", df['close_date'].isna().sum())

df['sales_cycle_days'] = (df['close_date'] - df['created_date']).dt.days
df = df[~(df['sales_cycle_days'] < 0)]

df.to_csv('sales_crm_clean.csv', index=False)
print(f"\nFinal shape: {df.shape}")
print("\nNull check:")
print(df[['created_date','close_date','sales_cycle_days']].isna().sum())

Null created_date after robust parse: 514
Null close_date after robust parse: 544

Final shape: (650, 13)

Null check:
created_date        514
close_date          544
sales_cycle_days    629
dtype: int64


In [14]:
import pandas as pd
from dateutil import parser as dateutil_parser
import numpy as np

# ── 1. Reload from raw ──────────────────────────────────────────────
df = pd.read_csv('sales_crm_raw.csv')
df = df.drop_duplicates(subset='deal_id', keep='first')

print("Loaded raw shape:", df.shape)
print("\nSample raw created_date values:")
print(df['created_date'].dropna().sample(15, random_state=1).tolist())

Loaded raw shape: (650, 12)

Sample raw created_date values:
['2022-09-29', '14/05/2023', '07-28-2023', 'June 14, 2023', '07-03-2022', '26 Dec 2022', '18 Nov 2023', '10-02-2022', '2023-04-25', '24/07/2023', '14 Dec 2023', '2022-09-15', '03-09-2022', '2023-11-23', '22 Aug 2022']


In [15]:
import pandas as pd
import numpy as np
from dateutil import parser as dateutil_parser

def parse_date_robust(val):
    if pd.isna(val) or str(val).strip() == '':
        return pd.NaT
    try:
        return dateutil_parser.parse(str(val).strip(), dayfirst=True)
    except Exception:
        return pd.NaT

# ── 1. Start clean from raw ─────────────────────────────────────────
df = pd.read_csv('sales_crm_raw.csv')
df = df.drop_duplicates(subset='deal_id', keep='first')
print("After dedup:", df.shape)

# ── 2. Clean stage ───────────────────────────────────────────────────
df['stage'] = df['stage'].str.strip().str.lower()
stage_mapping = {
    'won': 'Closed Won', 'closed won': 'Closed Won', 'wonnn': 'Closed Won',
    'lost': 'Closed Lost', 'closed lost': 'Closed Lost',
    'proposal sent': 'Proposal Sent', 'qualified': 'Qualified',
    'discovery': 'Discovery', 'negotiation': 'Negotiation',
    'prospecting': 'Prospecting'
}
df['stage'] = df['stage'].map(stage_mapping)

# ── 3. Clean region & lead_source ───────────────────────────────────
df['region'] = df['region'].str.strip().str.title()
region_mapping = {'East India': 'East', 'North': 'North',
                  'South': 'South', 'East': 'East', 'West': 'West'}
df['region'] = df['region'].map(region_mapping)
df['lead_source'] = df['lead_source'].str.strip().str.title()

# ── 4. Fix deal_value ────────────────────────────────────────────────
df['deal_value'] = pd.to_numeric(df['deal_value'], errors='coerce')
df['deal_value'] = df['deal_value'].abs()
df['deal_value'] = df.groupby(['industry', 'lead_source'])['deal_value'] \
                     .transform(lambda x: x.fillna(x.mean()))
df['deal_value'] = df['deal_value'].fillna(df['deal_value'].median())

# ── 5. Parse dates robustly ──────────────────────────────────────────
print("\nParsing dates (this may take 10-15 seconds)...")
df['created_date'] = df['created_date'].apply(parse_date_robust)
df['close_date']   = df['close_date'].apply(parse_date_robust)

print("Null created_date:", df['created_date'].isna().sum())
print("Null close_date:  ", df['close_date'].isna().sum())

# ── 6. Sales cycle & final cleanup ──────────────────────────────────
df['sales_cycle_days'] = (df['close_date'] - df['created_date']).dt.days
bad = (df['sales_cycle_days'] < 0).sum()
print(f"Negative sales cycle rows dropped: {bad}")
df = df[~(df['sales_cycle_days'] < 0)]

# ── 7. Save ──────────────────────────────────────────────────────────
df.to_csv('sales_crm_clean.csv', index=False)
print(f"\nFinal clean shape: {df.shape}")
print("\nColumn null summary:")
print(df.isnull().sum())
print("\nSample:")
print(df[['deal_id','stage','deal_value','region',
          'created_date','close_date','sales_cycle_days']].head(8).to_string())

After dedup: (650, 12)

Parsing dates (this may take 10-15 seconds)...
Null created_date: 0
Null close_date:   73
Negative sales cycle rows dropped: 52

Final clean shape: (598, 13)

Column null summary:
deal_id               0
company_name          0
industry              0
lead_source           0
deal_value            0
stage                 0
created_date          0
close_date           73
sales_rep             0
region                0
next_step           143
probability           0
sales_cycle_days     73
dtype: int64

Sample:
     deal_id          stage     deal_value region created_date close_date  sales_cycle_days
0  DEAL-0382    Negotiation  196753.320000  North   2022-06-11 2022-09-24             105.0
1  DEAL-0236     Closed Won   21486.270000   West   2022-03-11 2022-07-23             134.0
2  DEAL-0258  Proposal Sent  215097.600000  South   2023-03-18 2023-08-14             149.0
3  DEAL-0339     Closed Won  138042.520000   West   2023-08-07 2023-10-15              69.0
4 

# Metric 1 of 3 — Pipeline Value
- Pipeline Value is the total monetary value of all deals in the sales pipeline, and a B2B sales manager tracks it weekly to forecast revenue, monitor deal flow, and ensure enough opportunities exist to hit targets.

In [16]:
import pandas as pd

df = pd.read_csv('sales_crm_clean.csv')

# ── Pipeline Value by Stage ──────────────────────────────────────────
pipeline = df.groupby('stage')['deal_value'].agg(
    deal_count='count',
    total_value='sum',
    avg_deal_value='mean'
).round(2).sort_values('total_value', ascending=False)

print("=== PIPELINE VALUE BY STAGE ===")
print(pipeline.to_string())

# ── Total active pipeline (excluding closed) ─────────────────────────
active_stages = ['Prospecting', 'Discovery', 'Qualified',
                 'Proposal Sent', 'Negotiation']
active_pipeline = df[df['stage'].isin(active_stages)]['deal_value'].sum()
total_pipeline  = df['deal_value'].sum()
closed_won_val  = df[df['stage'] == 'Closed Won']['deal_value'].sum()
closed_lost_val = df[df['stage'] == 'Closed Lost']['deal_value'].sum()

print(f"\n=== PIPELINE SUMMARY ===")
print(f"Total Pipeline Value      : ₹{total_pipeline:,.0f}")
print(f"Active Pipeline Value     : ₹{active_pipeline:,.0f}")
print(f"Closed Won Revenue        : ₹{closed_won_val:,.0f}")
print(f"Closed Lost Value         : ₹{closed_lost_val:,.0f}")
print(f"Win Rate (by value)       : {(closed_won_val/(closed_won_val+closed_lost_val)*100):.1f}%")

=== PIPELINE VALUE BY STAGE ===
               deal_count  total_value  avg_deal_value
stage                                                 
Closed Won            148  18413463.85       124415.30
Closed Lost            87  15897267.34       182727.21
Negotiation            88  14676298.48       166776.12
Proposal Sent          77  14109292.47       183237.56
Discovery              54  11979032.11       221833.93
Prospecting            62  10326526.38       166556.88
Qualified              82   9540791.76       116351.12

=== PIPELINE SUMMARY ===
Total Pipeline Value      : ₹94,942,672
Active Pipeline Value     : ₹60,631,941
Closed Won Revenue        : ₹18,413,464
Closed Lost Value         : ₹15,897,267
Win Rate (by value)       : 53.7%


- Closed Lost value helps answer: “How much potential revenue are we losing — and where in the sales process are we failing to convert deals?”

# Metric 2 of 3 — Funnel Conversion

In [17]:
# ── Define funnel order ──────────────────────────────────────────────
funnel_order = ['Prospecting', 'Discovery', 'Qualified',
                'Proposal Sent', 'Negotiation', 'Closed Won']

funnel = df[df['stage'].isin(funnel_order)].groupby('stage').agg(
    deal_count=('deal_id', 'count'),
    total_value=('deal_value', 'sum')
).reindex(funnel_order)

# ── Conversion rate stage to stage ──────────────────────────────────
funnel['conversion_to_next'] = (
    funnel['deal_count'].shift(-1) / funnel['deal_count'] * 100
).round(1)

# ── Overall funnel conversion ────────────────────────────────────────
top_of_funnel    = funnel.loc['Prospecting', 'deal_count']
bottom_of_funnel = funnel.loc['Closed Won', 'deal_count']
overall_conv     = round(bottom_of_funnel / top_of_funnel * 100, 1)

print("=== FUNNEL CONVERSION ANALYSIS ===")
print(funnel.to_string())
print(f"\nOverall Funnel Conversion (Prospecting → Closed Won): {overall_conv}%")
print(f"\nBiggest drop-off stage:")
drop = funnel['conversion_to_next'].idxmin()
print(f"  {drop} → next stage: {funnel.loc[drop, 'conversion_to_next']}%")

=== FUNNEL CONVERSION ANALYSIS ===
               deal_count   total_value  conversion_to_next
stage                                                      
Prospecting            62  1.032653e+07                87.1
Discovery              54  1.197903e+07               151.9
Qualified              82  9.540792e+06                93.9
Proposal Sent          77  1.410929e+07               114.3
Negotiation            88  1.467630e+07               168.2
Closed Won            148  1.841346e+07                 NaN

Overall Funnel Conversion (Prospecting → Closed Won): 238.7%

Biggest drop-off stage:
  Prospecting → next stage: 87.1%


In [18]:
# ── Honest stage distribution ────────────────────────────────────────
stage_dist = df.groupby('stage').agg(
    deal_count=('deal_id', 'count'),
    total_value=('deal_value', 'sum'),
    avg_value=('deal_value', 'mean')
).round(2)

stage_dist['pct_of_deals'] = (stage_dist['deal_count'] /
                               stage_dist['deal_count'].sum() * 100).round(1)
stage_dist['pct_of_value'] = (stage_dist['total_value'] /
                               stage_dist['total_value'].sum() * 100).round(1)

print("=== STAGE DISTRIBUTION ANALYSIS ===")
print(stage_dist.sort_values('deal_count', ascending=False).to_string())

# ── Win/Loss ratio ───────────────────────────────────────────────────
won  = df[df['stage'] == 'Closed Won'].shape[0]
lost = df[df['stage'] == 'Closed Lost'].shape[0]
total_closed = won + lost

print(f"\n=== WIN / LOSS ANALYSIS ===")
print(f"Deals Won       : {won}")
print(f"Deals Lost      : {lost}")
print(f"Win Rate        : {won/total_closed*100:.1f}%")
print(f"Loss Rate       : {lost/total_closed*100:.1f}%")

# ── Win rate by region ───────────────────────────────────────────────
closed = df[df['stage'].isin(['Closed Won', 'Closed Lost'])].copy()
closed['is_won'] = (closed['stage'] == 'Closed Won').astype(int)
win_by_region = closed.groupby('region')['is_won'].agg(
    total='count', wins='sum'
)
win_by_region['win_rate_%'] = (win_by_region['wins'] /
                                win_by_region['total'] * 100).round(1)

print("\n=== WIN RATE BY REGION ===")
print(win_by_region.sort_values('win_rate_%', ascending=False).to_string())

=== STAGE DISTRIBUTION ANALYSIS ===
               deal_count  total_value  avg_value  pct_of_deals  pct_of_value
stage                                                                        
Closed Won            148  18413463.85  124415.30          24.7          19.4
Negotiation            88  14676298.48  166776.12          14.7          15.5
Closed Lost            87  15897267.34  182727.21          14.5          16.7
Qualified              82   9540791.76  116351.12          13.7          10.0
Proposal Sent          77  14109292.47  183237.56          12.9          14.9
Prospecting            62  10326526.38  166556.88          10.4          10.9
Discovery              54  11979032.11  221833.93           9.0          12.6

=== WIN / LOSS ANALYSIS ===
Deals Won       : 148
Deals Lost      : 87
Win Rate        : 63.0%
Loss Rate       : 37.0%

=== WIN RATE BY REGION ===
        total  wins  win_rate_%
region                         
South      61    40        65.6
West       69    4

# Metric 3 of 3 — Sales Velocity

In [19]:
# ── Sales Velocity Formula ───────────────────────────────────────────
# Sales Velocity = (Number of Deals × Avg Deal Value × Win Rate) / Avg Sales Cycle

won_deals = df[df['stage'] == 'Closed Won'].copy()

num_deals    = len(won_deals)
avg_value    = won_deals['deal_value'].mean()
win_rate     = num_deals / (num_deals + len(df[df['stage'] == 'Closed Lost']))
avg_cycle    = won_deals['sales_cycle_days'].dropna().mean()

sales_velocity = (num_deals * avg_value * win_rate) / avg_cycle

print("=== SALES VELOCITY ANALYSIS ===")
print(f"Number of Won Deals       : {num_deals}")
print(f"Avg Deal Value            : ₹{avg_value:,.0f}")
print(f"Win Rate                  : {win_rate*100:.1f}%")
print(f"Avg Sales Cycle (days)    : {avg_cycle:.1f}")
print(f"\nSales Velocity            : ₹{sales_velocity:,.0f} per day")
print(f"Monthly Revenue Run Rate  : ₹{sales_velocity*30:,.0f}")
print(f"Annual Revenue Run Rate   : ₹{sales_velocity*365:,.0f}")

# ── Sales velocity by region ─────────────────────────────────────────
print("\n=== SALES VELOCITY BY REGION ===")
for region in df['region'].dropna().unique():
    r_df     = df[df['region'] == region]
    r_won    = r_df[r_df['stage'] == 'Closed Won']
    r_lost   = r_df[r_df['stage'] == 'Closed Lost']
    if len(r_won) < 5:
        continue
    r_num    = len(r_won)
    r_val    = r_won['deal_value'].mean()
    r_wr     = r_num / (r_num + len(r_lost)) if (r_num + len(r_lost)) > 0 else 0
    r_cycle  = r_won['sales_cycle_days'].dropna().mean()
    r_vel    = (r_num * r_val * r_wr) / r_cycle if r_cycle > 0 else 0
    print(f"  {region:<8} → ₹{r_vel:,.0f}/day  |  Avg Cycle: {r_cycle:.0f} days  |  Win Rate: {r_wr*100:.1f}%")

=== SALES VELOCITY ANALYSIS ===
Number of Won Deals       : 148
Avg Deal Value            : ₹124,415
Win Rate                  : 63.0%
Avg Sales Cycle (days)    : 124.0

Sales Velocity            : ₹93,553 per day
Monthly Revenue Run Rate  : ₹2,806,605
Annual Revenue Run Rate   : ₹34,147,022

=== SALES VELOCITY BY REGION ===
  North    → ₹33,956/day  |  Avg Cycle: 105 days  |  Win Rate: 63.2%
  West     → ₹25,894/day  |  Avg Cycle: 142 days  |  Win Rate: 63.8%
  South    → ₹26,722/day  |  Avg Cycle: 119 days  |  Win Rate: 65.6%
  East     → ₹9,011/day  |  Avg Cycle: 137 days  |  Win Rate: 56.8%


In [1]:
import os
print(os.getcwd())

C:\Users\mahaj\Portfolio\RevOps_Sales_Analytics_Project


In [2]:
import os
print(os.getcwd())
print(os.path.exists('sales_crm_clean.csv'))

C:\Users\mahaj\Portfolio\RevOps_Sales_Analytics_Project
True
